# DAAAnalyzer_2.0.AnalyzeABRs.ipynb
Created by: JFranco | Created On: 27 NOV 2024 | Last Env: QISHA | Last Run Date: 27 NOV 2024

This Python notebook will read in all DPOAE files for a specific DAA Run as specified by the user. It will make a lineplot for all availble files and label them by category. 

In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
import glob
import os
import xlrd
import math
import matplotlib.pyplot as plt

In [11]:
#                *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Key identifiers
prepID = 'CMI_003'
daaRun = '1'
daaType = 'Pre-Noise Exposure'
isoFile = 'IsoDP-'+daaRun+'-1.tsv'
fnMD = 'CMI_003.Metadata.AnimalInfo.csv'
fileFiltered = "ABR-1-1.tsv-filtered.txt"
fileAnalyzed = "ABR-1-1.tsv-analyzed.txt"

# Directories 
#   existing ones 
dirMain = '/Users/joyfranco/Dropbox (Partners HealthCare)/JF_Shared/Data/CFTS/'
dirPrep = dirMain+prepID+'/'
dirData = dirPrep+'RawData/'
dirMD = dirPrep+'Metadata/'
pathTest = dirData+prepID+".C016.1/"

#   ones that need to be made    
dirSV = dirPrep+'Results/'        

In [12]:
df = pd.read_csv(pathTest+fileFiltered, delimiter='\t', header=0) 
df

,Time (ms),20.0 dB,25.0 dB,30.0 dB,35.0 dB,40.0 dB,45.0 dB,50.0 dB,55.0 dB,60.0 dB
0,0.00,-0.025547,0.008846,-0.073541,-0.057114,0.009052,-0.003221,-0.072090,-0.045683,0.027824
1,0.04,-0.047892,0.013597,-0.110884,-0.017521,0.004125,-0.014425,-0.107239,-0.076539,-0.018915
2,0.08,-0.059329,0.024015,-0.141212,0.002861,-0.004226,-0.021701,-0.133697,-0.099652,-0.043134
3,0.12,-0.067238,0.051110,-0.161291,0.000600,-0.013252,-0.021316,-0.150656,-0.114893,-0.047406
4,0.16,-0.080206,0.086448,-0.174117,-0.028970,-0.017787,-0.017246,-0.160801,-0.118011,-0.035823
...,...,...,...,...,...,...,...,...,...,...
208,8.32,-0.025066,0.052784,0.108355,-0.059496,0.007201,-0.019352,0.005253,-0.148231,0.230071
209,8.36,-0.025428,0.043189,0.108422,-0.041101,0.017096,0.002066,0.005989,-0.109736,0.225133
210,8.40,-0.016859,0.034102,0.085038,-0.031977,0.014460,0.011595,-0.001594,-0.094181,0.203902
211,8.44,-0.006861,0.030745,0.062501,-0.033138,0.009868,0.002486,-0.007292,-0.097996,0.174892


In [22]:
df = pd.read_csv(pathTest+fileAnalyzed, sep="\t", skiprows=6) 
df

,Level,0.3msec Avg,0.3msec StDev,P1 Latency,P1 Amplitude,N1 Latency,N1 Amplitude,P2 Latency,P2 Amplitude,N2 Latency,...,P4 Latency,P4 Amplitude,N4 Latency,N4 Amplitude,P5 Latency,P5 Amplitude,N5 Latency,N5 Amplitude,CorrCoef,Unnamed: 24
0,60.0,-0.019814,0.024346,1.76,0.50,2.32,-0.22,2.56,0.14,2.68,...,4.56,0.51,5.44,-0.25,5.76,-0.02,6.44,-0.27,NaN,NaN
1,55.0,-0.100166,0.027089,1.84,0.38,2.28,-0.11,2.68,0.25,3.12,...,4.68,0.27,5.52,-0.24,6.04,0.10,6.60,-0.18,0.666,NaN
2,50.0,-0.137518,0.033659,1.96,0.35,2.36,-0.20,2.76,0.21,3.24,...,4.68,0.11,5.44,-0.20,6.24,0.13,6.56,0.00,0.721,NaN
3,45.0,-0.014788,0.005835,2.00,0.15,2.32,-0.07,2.80,0.18,3.28,...,4.88,0.24,5.44,-0.06,6.28,0.09,6.56,-0.04,0.574,NaN
4,40.0,-0.012166,0.015355,2.20,0.15,2.52,-0.05,2.96,0.09,3.28,...,5.08,0.13,5.56,-0.12,6.28,-0.15,6.56,-0.14,0.144,NaN
5,35.0,-0.031259,0.025727,-2.20,0.16,-2.68,-0.04,-2.96,0.03,-3.16,...,-5.08,0.06,-5.40,-0.09,-6.60,0.05,-6.96,-0.02,0.199,NaN
6,30.0,-0.146575,0.038171,-2.20,0.10,-2.56,-0.06,-3.44,0.13,-3.44,...,-5.08,-0.12,-5.40,-0.07,-7.08,0.08,-7.80,-0.08,0.540,NaN
7,25.0,0.061144,0.045714,-2.32,0.10,-2.56,-0.01,-3.44,0.03,-3.44,...,-5.36,0.11,-5.40,0.10,-7.08,0.00,-7.80,0.09,-0.047,NaN
8,20.0,-0.062474,0.019013,-2.60,0.19,-3.20,-0.01,-3.52,0.16,-3.52,...,-5.36,-0.10,-5.40,-0.11,-7.36,-0.06,-7.60,-0.14,0.237,NaN


In [5]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfMD = pd.read_csv(dirMD+fnMD)
dfMD.reset_index(inplace=True)
dfMD

,index,AnimalID,Sex,DOB,CMIID,InjectionDate,AgeAtInjection,Package,InjGroup,NE,NEDate,AgeAtNE,NELevel,NEBand,NEDuration,NEGroup
0,0,C016,M,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,1
1,1,C017,F,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,2
2,2,C018,M,9/24/24,CMI_003,11/11/24,7,AAV,HABclw,Yes,11/25/24,9,97.5,8_16,2h,2
3,3,C019,F,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,1
4,4,C020,M,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,1
5,5,C021,F,9/24/24,CMI_003,11/11/24,7,AAV,CtrlInj,Yes,11/25/24,9,97.5,8_16,2h,2
6,6,C022,M,9/24/24,CMI_003,11/11/24,7,AAV,UnInj,Yes,11/25/24,9,97.5,8_16,2h,2
7,7,C023,F,9/24/24,CMI_003,11/11/24,7,AAV,UnInj,Yes,11/25/24,9,97.5,8_16,2h,1


In [6]:
#           *** INITIALIZE RUN SPECIFIC DIRECTORY ETC FOR STORING RESULTS **
# Create directory for storing spreadsheetS and summary plotS for this run
if not os.path.exists(dirSV): os.mkdir(dirSV)

In [7]:
#           *** GENERATE A LIST OF DAA FOLDERS THAT WILL BE ANALYZED **
# Get a list of all the folders that match the prep
allData = os.listdir(dirData)
dataList = []
for datum in allData:
    if ('.'+daaRun) in datum:
        dataList.append(datum)
dataList

['CMI_003.C020.1',
 'CMI_003.C023.1',
 'CMI_003.C016.1',
 'CMI_003.C018.1',
 'CMI_003.C022.1',
 'CMI_003.C017.1',
 'CMI_003.C019.1',
 'CMI_003.C021.1']

In [20]:
#           *** ITERATE THROUGH FOLDERS AND GENERATE ABR PLOTS **
# 1. Open the folder and get all available ABR files
for daa in dataList:
    # Setup the path to the file
    abrFold = dirData+daa+'/'
    # Get a list of available ABR files
    allFiles = os.listdir(abrFold)
    files = []
    for file in allFiles:
        if ("ABR-"+daaRun) in file:
            files.append(file)
    # Go through the files and make a line plot that contains all ABRs for that frequency
    for file in files:
        # Setup the path to the file
        path = dirData+daa+'/'+file
        # Read in the file & make adjustments for readability
        dfABR = pd.read_csv(path, sep='\t', encoding='unicode_escape', lineterminator='\n')
        # Have to get out:
        freq = dfABR.columns.tolist()[6][8:-1]
        # Levels
        # Number of averages -- for each column showing 

In [48]:
str = dfABR.iloc[0].values[0][7:-1]

In [49]:
tries = [i.strip() for i in str.split(';')]
tries

['554', '566', '558', '552', '250']

In [52]:
# This one is going to be harder because the levels are embedded and it'll be hard to say how the indexing changes.
# Should pln on just searching for the word LEVELS: and then get all of the characters until the next "\r"
dfABR.iloc[1].values[0]

array(['\r:CFTS Version: V2.37.2 (R3282)\r:LEVELS:20;25;30;35;40;\r:DATA\r -0.050935',
       0.074373, -0.064534, 0.03543, -0.020103, nan, nan, nan, nan, nan,
       nan, nan], dtype=object)